In [1]:
import pandas as pd
from sklearn.utils import shuffle
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler

In [2]:
def split_set(dfs):
    big_df = pd.concat(dfs)
    big_df = shuffle(big_df)
    test_len = len(big_df) // 5
    test_set = big_df[:test_len]
    train_set = big_df[test_len+1:]
    return test_set, train_set

In [3]:
esa_21_22_fast_actions_parsed = pd.read_pickle('esa_21_22_fast_actions_parsed.pkl')
esa_22_23_fast_actions_parsed = pd.read_pickle('esa_22_23_fast_actions_parsed.pkl')
len(esa_21_22_fast_actions_parsed)

22126

In [4]:
len(esa_22_23_fast_actions_parsed)

20632

In [5]:
test_set, train_set = split_set([esa_21_22_fast_actions_parsed, esa_22_23_fast_actions_parsed])
print(len(test_set))
len(train_set)

8551


34206

In [6]:
test_set['was_shot'] = [1 if (shots > 0) else 0 for shots in test_set['num_of_shot'].values]
len(test_set[test_set['was_shot'] == 1])

301

In [7]:
train_set['was_shot'] = [1 if (shots > 0) else 0 for shots in train_set['num_of_shot'].values]
len(train_set[train_set['was_shot'] == 1])

1214

In [8]:
# def set_good_end_values(df):
#     opponent_penalty_area = ['K3', 'K4', 'K5', 'K6', 'L3', 'L4', 'L5', 'L6']
#     newDf = df
#     newDf['good_end'] = [1 if (end_area in opponent_penalty_area) or (shots > 0) else 0 for end_area, shots in zip(df['end_area'].values, df['num_of_shot'].values)]
#     return newDf
    

In [9]:
# euro_matches = set_good_end_values(euro_matches)
# fifa_matches = set_good_end_values(fifa_matches)
# len(euro_matches[euro_matches['good_end'] == 1])

In [10]:
def get_sets_for_model(df_train, df_test):
    df_train = shuffle(df_train)
    df_test = shuffle(df_test)

    # scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))

    # df_train_ids = df_train['id'].values  # type: ignore
    # df_test_ids = df_test['id'].values  # type: ignore
    # df_train_teams = df_train['team'].values  # type: ignore
    # df_test_teams = df_test['team'].values  # type: ignore
    # df_train_was_shot = df_train['was_shot'].values  # type: ignore
    # df_test_was_shot = df_test['was_shot'].values  # type: ignore
    # # df_train_good_ends = df_train['good_end'].values  # type: ignore
    # # df_test_good_ends = df_test['good_end'].values  # type: ignore

    # df_train = df_train.drop(columns=['id', 'team', 'path', 'start_area', 'end_area', 'was_shot']) # type: ignore
    # df_test = df_test.drop(columns=['id', 'team', 'path', 'start_area', 'end_area', 'was_shot']) # type: ignore

    df_train_columns = df_train.columns.sort_values()
    df_test_columns = df_test.columns.sort_values()

    # df_train = scaler.fit_transform(df_train)
    # df_test = scaler.fit_transform(df_test)

    df_train = pd.DataFrame(df_train, columns=df_train_columns)
    df_test = pd.DataFrame(df_test, columns=df_test_columns)

    # df_train['id'] = df_train_ids
    # df_test['id'] = df_test_ids
    # df_train['team'] = df_train_teams
    # df_test['team'] = df_test_teams
    # df_train['was_shot'] = df_train_was_shot
    # df_test['was_shot'] = df_test_was_shot

    y_train = df_train['was_shot']
    X_train = df_train.drop(columns=['was_shot'])

    y_test = df_test['was_shot']
    X_test = df_test.drop(columns=['was_shot'])

    return X_train, X_test, y_train, y_test

In [11]:
X_train, X_test, y_train, y_test = get_sets_for_model(train_set, test_set)

In [12]:
from xgboost import XGBClassifier

def train_model(X_train, X_test, y_train):
    model = XGBClassifier(objective='binary:logistic')

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    model.fit(input_X_train, y_train)

    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    XGB_predictions = model.predict(input_X_test)

    return model, XGB_predictions, input_X_test

model, pred, input_test = train_model(X_train, X_test, y_train)

In [13]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def evaluate_model(y_test, pred):
    print('Confusion matrix')
    print(confusion_matrix(y_test, pred))
    print('Classification report')
    print(classification_report(y_test, pred))
    print('Accuracy= ', accuracy_score(y_test, pred))

evaluate_model(y_test, pred)

Confusion matrix
[[8240   10]
 [  34  267]]
Classification report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8250
           1       0.96      0.89      0.92       301

    accuracy                           0.99      8551
   macro avg       0.98      0.94      0.96      8551
weighted avg       0.99      0.99      0.99      8551

Accuracy=  0.9948544029938019


In [14]:
from datetime import datetime

# datetime object containing current date and time
now = datetime.now()
 
print("now =", now)

# dd/mm/YY H:M:S
dt_string = now.strftime("%d_%m_%Y_%H-%M-%S")
print("date and time =", dt_string)

now = 2023-06-11 18:35:38.927155
date and time = 11_06_2023_18-35-38


In [15]:
def feature_importances_raport(input_X_test, model, set_title):
    result = pd.DataFrame(columns=['name', 'value'])
    result['name'] = input_X_test.columns
    result['value'] = model.feature_importances_

    result = result.sort_values(by=['value'], ascending=False)

    result.to_csv(f'features_{set_title}.csv')
    print(result)
    return result

feature_importances = feature_importances_raport(input_test, model, f'fast_sequences_experiment_{dt_string}')

                    name     value
1                  end_x  0.342634
2                  end_y  0.077758
4   num_of_ball_receipt*  0.076302
15     num_of_miscontrol  0.071701
9    num_of_dispossessed  0.062522
13         num_of_events  0.048004
16           num_of_pass  0.046431
5   num_of_ball_recovery  0.035956
8       num_of_clearance  0.034353
14   num_of_interception  0.034309
10        num_of_dribble  0.029606
11           num_of_duel  0.027755
17    opponent_distances  0.025261
7           num_of_carry  0.023769
6           num_of_block  0.014773
3           key_pass_len  0.013539
18               start_x  0.012178
0               duration  0.011610
19               start_y  0.011540
12          num_of_error  0.000000


In [16]:
def generate_scores(model, X_test, X_train, y_train, set_title):
    model = XGBClassifier(objective='binary:logistic')

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    model.fit(input_X_train, y_train)
    
    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    model = model.predict_proba(input_X_test)

    X_test['score_positive'] = [m[1] for m in model]
    X_test['score_negative'] = [m[0] for m in model]

    X_test.to_csv(f'{set_title}_evaluated.csv')

    return X_test

df_with_scores = generate_scores(model, X_test, X_train, y_train, f'fast_sequences_experiment_{dt_string}')

In [17]:
import statistics
import pandas as pd

def top_teams_by_median(df_with_scores, dt_string):
    teams = list(set(df_with_scores['team'].values))
    df_team_scores = pd.DataFrame()
    df_scores = []
    df_teams = []
    df_obv = []
    df_xg = []
    for team in teams:
        score = statistics.median(df_with_scores[df_with_scores['team'] == team]['score_positive'].values)
        obv = statistics.median(df_with_scores[df_with_scores['team'] == team]['obv_for'].values)
        xg = statistics.median(df_with_scores[df_with_scores['team'] == team]['xg'].values)
        df_teams.append(team)
        df_obv.append(obv)
        df_xg.append(xg)
        df_scores.append(score)
    df_team_scores['team'] = df_teams
    df_team_scores['score'] = df_scores
    df_team_scores['obv'] = df_obv
    df_team_scores['xg'] = df_xg
    result = df_team_scores.sort_values(by=['score'], ascending=False)
    result.to_csv(f'fast_sequences_top_teams_mean_{dt_string}.csv')
    return result

def top_teams_by_mean(df_with_scores, dt_string):
    teams = list(set(df_with_scores['team'].values))
    df_team_scores = pd.DataFrame()
    df_scores = []
    df_teams = []
    df_obv = []
    df_xg = []
    for team in teams:
        score = statistics.mean(df_with_scores[df_with_scores['team'] == team]['score_positive'].values)
        obv = statistics.mean(df_with_scores[df_with_scores['team'] == team]['obv_for'].values)
        xg = statistics.mean(df_with_scores[df_with_scores['team'] == team]['xg'].values)
        df_teams.append(team)
        df_obv.append(obv)
        df_xg.append(xg)
        df_scores.append(score)
    df_team_scores['team'] = df_teams
    df_team_scores['score'] = df_scores
    df_team_scores['obv'] = df_obv
    df_team_scores['xg'] = df_xg
    result = df_team_scores.sort_values(by=['score'], ascending=False)
    result.to_csv(f'fast_sequences_top_teams_median_{dt_string}.csv')
    return result

def top_teams_by_sum(df_with_scores, dt_string):
    teams = list(set(df_with_scores['team'].values))
    df_team_scores = pd.DataFrame()
    df_scores = []
    df_teams = []
    df_obv = []
    df_xg = []
    for team in teams:
        score = sum(df_with_scores[df_with_scores['team'] == team]['score_positive'])
        obv = sum(df_with_scores[df_with_scores['team'] == team]['obv_for'])
        xg = sum(df_with_scores[df_with_scores['team'] == team]['xg'])
        df_teams.append(team)
        df_obv.append(obv)
        df_xg.append(xg)
        df_scores.append(score)
    df_team_scores['team'] = df_teams
    df_team_scores['score'] = df_scores
    df_team_scores['obv'] = df_obv
    df_team_scores['xg'] = df_xg
    result = df_team_scores.sort_values(by=['score'], ascending=False)
    result.to_csv(f'fast_sequences_top_teams_sum_{dt_string}.csv')
    return result

In [18]:
top_teams_by_sum(df_with_scores, dt_string)

,team,score,obv,xg
4,Lech Poznan,25.355795,3.833369,2.729455
1,Zagłębie Lubin,25.124717,4.361057,2.912748
2,Pogoń Szczecin,24.443983,3.894117,3.897946
14,Lechia Gdańsk,21.039369,4.452083,3.171253
13,Śląsk Wrocław,19.319424,2.881633,1.843239
11,Górnik Zabrze,19.084599,3.627208,4.612264
18,Cracovia Kraków,16.525891,2.545324,0.873356
16,Jagiellonia Białystok,15.933372,3.188979,2.096613
10,Raków Częstochowa,13.275062,4.234369,2.732553
15,Piast Gliwice,12.463320,3.147348,1.336042


In [19]:
top_teams_by_mean(df_with_scores, dt_string)

,team,score,obv,xg
4,Lech Poznan,0.062453,0.009442,0.006723
2,Pogoń Szczecin,0.059043,0.009406,0.009415
1,Zagłębie Lubin,0.050860,0.008828,0.005896
14,Lechia Gdańsk,0.048145,0.010188,0.007257
13,Śląsk Wrocław,0.040758,0.006079,0.003889
11,Górnik Zabrze,0.035211,0.006692,0.008510
16,Jagiellonia Białystok,0.032650,0.006535,0.004296
7,Widzew Łódź,0.030943,0.003574,0.001907
9,LKS Termalica Bruk-Bet Nieciecza,0.030486,0.002912,0.004018
18,Cracovia Kraków,0.029776,0.004586,0.001574


In [20]:
top_teams_by_median(df_with_scores, dt_string)

,team,score,obv,xg
4,Lech Poznan,0.000029,0.001752,0.0
2,Pogoń Szczecin,0.000028,0.001519,0.0
20,Korona Kielce,0.000027,0.002220,0.0
9,LKS Termalica Bruk-Bet Nieciecza,0.000025,0.001361,0.0
3,Stal Mielec,0.000024,0.001626,0.0
12,Wisła Kraków,0.000024,0.001865,0.0
10,Raków Częstochowa,0.000023,0.001675,0.0
8,Wisła Płock,0.000023,0.001780,0.0
14,Lechia Gdańsk,0.000023,0.001666,0.0
11,Górnik Zabrze,0.000023,0.001564,0.0


In [21]:
def show_correlation(df_with_scores, dt_string):
    corr_with_xg = df_with_scores['score_positive'].corr(df_with_scores['xg'])
    corr_with_obv = df_with_scores['score_positive'].corr(df_with_scores['obv_for'])
    corr_with_obv_total = df_with_scores['score_positive'].corr(df_with_scores['obv_total'])
    print(f'Correlation with xg: {corr_with_xg:.3f}')
    print(f'Correlation with obv_for: {corr_with_obv:.3f}')
    print(f'Correlation with obv_total: {corr_with_obv_total:.3f}')
    result = pd.DataFrame()
    result['xg_corr'] = corr_with_xg
    result['obv_corr'] = corr_with_obv
    result['obv_total'] = corr_with_obv_total
    result.to_csv(f'fast_sequences_correlations_{dt_string}.csv')

In [22]:
show_correlation(df_with_scores, dt_string)

Correlation with xg: 0.563
Correlation with obv_for: 0.394
Correlation with obv_total: 0.394
